In [2]:
from src.components.data_processing.data_loader import DataLoader, DataUtils
from loguru import logger
import polars.selectors as cs
import polars as pl

In [3]:
import os
import s3fs

os.environ["AWS_ACCESS_KEY_ID"] = "KGBESCEJ2L4ANCRPM0UB"
os.environ["AWS_SECRET_ACCESS_KEY"] = "GYb0HPOP1vKciDq8qOunyNTjGz+HD67XF3+4dd+6"
os.environ["AWS_SESSION_TOKEN"] = (
    "eyJhbGciOiJIUzUxMiIsInR5cCI6IkpXVCJ9.eyJhY2Nlc3NLZXkiOiJLR0JFU0NFSjJMNEFOQ1JQTTBVQiIsImFsbG93ZWQtb3JpZ2lucyI6WyIqIl0sImF1ZCI6WyJtaW5pby1kYXRhbm9kZSIsIm9ueXhpYSIsImFjY291bnQiXSwiYXV0aF90aW1lIjoxNzc4NDg0MjcxLCJhenAiOiJvbnl4aWEiLCJjbmYiOnsiamt0IjoiSE51MEVHcHVuQ3k1emRjcmhaWExGUWF0TjRPSVNFbWZtZnk1a3k3MlBINCJ9LCJlbWFpbCI6ImFydGh1ci5tYW5jZWF1QHN0dWRlbnQtY3MuZnIiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZXhwIjoxNzc5MDg5Mjc1LCJmYW1pbHlfbmFtZSI6Ik1hbmNlYXUiLCJnaXZlbl9uYW1lIjoiQXJ0aHVyIiwiZ3JvdXBzIjpbIlVTRVJfT05ZWElBIl0sImlhdCI6MTc3ODQ4NDQ3NSwiaXNzIjoiaHR0cHM6Ly9hdXRoLmxhYi5zc3BjbG91ZC5mci9hdXRoL3JlYWxtcy9zc3BjbG91ZCIsImp0aSI6Im9ucnRydDo0MTNkMDc4OS1mNjgxLTBjY2QtZTg5My1mYmY1YjU0OTkxZWQiLCJsb2NhbGUiOiJlbiIsIm5hbWUiOiJBcnRodXIgTWFuY2VhdSIsInBvbGljeSI6InN0c29ubHkiLCJwcmVmZXJyZWRfdXNlcm5hbWUiOiJhcnRodXJtYW5jZWF1IiwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXNzcGNsb3VkIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwicm9sZXMiOlsib2ZmbGluZV9hY2Nlc3MiLCJ1bWFfYXV0aG9yaXphdGlvbiIsImRlZmF1bHQtcm9sZXMtc3NwY2xvdWQiXSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBncm91cHMgZW1haWwiLCJzaWQiOiJOQUk2UGw1VUwyd2VMazZCeHN4NElwTWsiLCJzdWIiOiJiYzc3Yjk3YS1lN2VkLTQ4ZWMtYmQ4Zi1lNzBjYjZhMTYyZGIiLCJ0eXAiOiJEUG9QIn0.ufLP0JGLNw9HdBPX1qbnZH9Q2gayd3iCfBR8n5VtamXHksBXtIsTSYc36C9UKn3Ugx1858v6BaxhprpwLWmmvw"
)
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + "minio.lab.sspcloud.fr"},
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    token=os.environ["AWS_SESSION_TOKEN"],
)

FIND a communes file with all the communes that existed and its history

In [3]:
communes = pl.scan_csv(
    "s3://arthurmanceau/election_modeling_uhcp/data/raw/insee_geo/2026/communes_2026.csv",
    storage_options=storage_options,
    infer_schema_length=40000,
).collect()

In [4]:
communes = communes.with_columns(
    DEP_NUM=pl.col("DEP").replace({"2A": 20, "2B": 20}).cast(pl.Int64),
    codecommune=pl.col("COM").cast(pl.String).str.zfill(5),
    codecommune_parent=pl.col("COMPARENT").cast(pl.String).str.zfill(5),
)

In [5]:
communes

TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT,DEP_NUM,codecommune,codecommune_parent
str,str,i64,str,str,str,i64,str,str,str,str,i64,i64,str,str
"""COM""","""1001""",84,"""1""","""01D""","""12""",5,"""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""","""L'Abergement-Clémenciat""","""108""",null,1,"""01001""",null
"""COM""","""1002""",84,"""1""","""01D""","""11""",5,"""ABERGEMENT DE VAREY""","""Abergement-de-Varey""","""L'Abergement-de-Varey""","""101""",null,1,"""01002""",null
"""COM""","""1004""",84,"""1""","""01D""","""11""",1,"""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""","""Ambérieu-en-Bugey""","""101""",null,1,"""01004""",null
"""COM""","""1005""",84,"""1""","""01D""","""12""",1,"""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""","""Ambérieux-en-Dombes""","""122""",null,1,"""01005""",null
"""COM""","""1006""",84,"""1""","""01D""","""11""",1,"""AMBLEON""","""Ambléon""","""Ambléon""","""104""",null,1,"""01006""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""COM""","""97613""",6,"""976""","""976R""",null,0,"""M TSANGAMOUJI""","""M'Tsangamouji""","""M'Tsangamouji""","""97613""",null,976,"""97613""",null
"""COM""","""97614""",6,"""976""","""976R""",null,1,"""OUANGANI""","""Ouangani""","""Ouangani""","""97610""",null,976,"""97614""",null
"""COM""","""97615""",6,"""976""","""976R""",null,0,"""PAMANDZI""","""Pamandzi""","""Pamandzi""","""97611""",null,976,"""97615""",null


In [6]:
data_path = "s3://arthurmanceau/election_modeling_uhcp/data/"
elections_to_exclude = []

In [ ]:
def interpret_election_path(relative_path):
    path_element = relative_path.split(os.sep)[
        -3:
    ]  # ['legislative', '1848', 'leg1848_csv']
    election_type = path_element[0]  # 'legislative'
    year = path_element[1]  # '1848'
    election_code = path_element[2].split("_")[0]  # leg1848
    return election_type, year, election_code


electoral_schema = {
    "dep": pl.String,
    "nomdep": pl.String,
    "codecommune": pl.String,
    "nomcommune": pl.String,
    "year": pl.Int64,
    "type": pl.String,
    "election_code": pl.String,
    "inscrits": pl.Float64,
    "votants": pl.Float64,
    "exprimes": pl.Float64,
    # "inscritsT2": pl.Float64,
    # "votantsT2": pl.Float64,
    # "exprimesT2": pl.Float64,
    "voteG": pl.Float64,
    "voteCG": pl.Float64,
    "voteC": pl.Float64,
    "voteCD": pl.Float64,
    "voteD": pl.Float64,
    "voteTG": pl.Float64,
    "voteTD": pl.Float64,
    "voteGCG": pl.Float64,
    "voteDCD": pl.Float64,
    "voixOUI": pl.Float64,
    "voixNON": pl.Float64,
}


def load_electoral_data():
    """Function that loads all electoral dataset. A schema is defined for all elections."""
    folder_path = os.path.join(data_path + "raw/", "elections/")
    combined = None
    xs = DataUtils._create_fs() if DataUtils._detect_s3(data_path) else os
    for root, dirs, files in xs.walk(folder_path):
        for dir in dirs:
            for root, _, files in xs.walk(os.path.join(folder_path, dir)):
                for file in files:
                    if (
                        file.endswith(".parquet")
                        and (not file.startswith("."))
                        and file not in elections_to_exclude
                    ):
                        election_type, year, election_code = interpret_election_path(
                            os.path.relpath(root, folder_path)
                        )
                        logger.debug(f"Processing election : {(election_type, year)}")

                        file_path = DataUtils.path_helper(
                            folder_path, os.path.join(root, file)
                        )
                        df = pl.scan_parquet(file_path).collect()

                        df = df.with_columns(
                            year=pl.lit(year).cast(pl.Int64),
                            election_code=pl.lit(election_code),
                            type=pl.lit(election_type),
                            codecommune=pl.col("codecommune")
                            .cast(pl.String)
                            .str.zfill(5),
                        )
                        df = df.cast(
                            {
                                key: value
                                for key, value in electoral_schema.items()
                                if key in df.columns
                            }
                        )
                        df = df.match_to_schema(
                            electoral_schema,
                            missing_columns="insert",
                            extra_columns="ignore",
                        )
                        combined = df if combined is None else combined.vstack(df)

    return combined.rechunk()


socio_economic_schema = {
    "codecommune": pl.String,  # key
    "raw": pl.Float64,
    "variable": pl.String,
    "year": pl.Int64,  # key
    "feature_name": pl.String,  # key
    "lag": pl.Float64,
    "rank": pl.Float64,
    "delta": pl.Float64,
    "pct_change": pl.Float64,
}


def load_socio_economic_data():
    folder_path = os.path.join(data_path, "raw/")
    combined = None
    xs = DataUtils._create_fs() if DataUtils._detect_s3(data_path) else os
    for root, dirs, files in xs.walk(folder_path):
        for dir in dirs:
            if dir != "elections":
                for root, _, files in xs.walk(os.path.join(folder_path, dir)):
                    for file in files:
                        if file.endswith(".parquet") and (not file.startswith(".")):
                            # Only consider the relevant file : commune level
                            # How to integrate departemental data?
                            key = "codecommune"
                            if "communes" not in file:
                                continue

                            if file[:5] == "codes":
                                continue

                            file_path = DataUtils.path_helper(
                                folder_path, os.path.join(root, file)
                            )
                            data_code = file_path.split("/")[-1].split(".")[0]
                            logger.debug(f"Processing file : {data_code}")

                            df = pl.scan_parquet(file_path).collect()

                            # Identify features (time evolution)
                            all_feature_years = df.select(
                                cs.matches(r".*\d{4}$")
                            ).columns

                            # Single unpivot for all features
                            df_long = (
                                df.unpivot(
                                    on=all_feature_years,
                                    index=key,
                                    variable_name="variable",
                                    value_name="raw",
                                )
                                .with_columns(
                                    pl.col("raw").cast(pl.Float64, strict=False),
                                    year=pl.col("variable").str.tail(4).cast(pl.Int32),
                                    feature_name=pl.col("variable").str.head(-4),
                                )
                                .sort([key, "feature_name", "year"])
                                .with_columns(
                                    lag=pl.col("raw")
                                    .shift(1)
                                    .over(key, "feature_name"),
                                    rank=pl.col("raw")
                                    .rank("dense")
                                    .over(key, "feature_name"),
                                    delta=pl.col("raw")
                                    .diff(1)
                                    .over(key, "feature_name"),
                                    pct_change=pl.col("raw")
                                    .pct_change(1)
                                    .over(key, "feature_name"),
                                )
                                .fill_nan(None)
                            )
                            float_cols = [
                                c
                                for c, dtype in zip(df_long.columns, df_long.dtypes)
                                if dtype in (pl.Float32, pl.Float64)
                            ]
                            df_long = df_long.with_columns(
                                [
                                    pl.when(pl.col(c).is_infinite())
                                    .then(None)
                                    .otherwise(pl.col(c))
                                    .alias(c)
                                    for c in float_cols
                                ]
                            )
                            df_long = df_long.cast(
                                {
                                    key: value
                                    for key, value in socio_economic_schema.items()
                                    if key in df_long.columns
                                }
                            )
                            df_long = df_long.match_to_schema(
                                socio_economic_schema,
                                missing_columns="insert",
                                extra_columns="ignore",
                            )
                            combined = (
                                df_long
                                if combined is None
                                else combined.vstack(df_long)
                            )

    return combined.rechunk().fill_nan(None)

In [356]:
election_datasets = load_electoral_data()

2026-04-30 10:31:31.793 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1848')
2026-04-30 10:31:32.026 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1849')
2026-04-30 10:31:32.249 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1850')
2026-04-30 10:31:32.462 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1851')
2026-04-30 10:31:32.750 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1871')
2026-04-30 10:31:32.979 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1872')
2026-04-30 10:31:33.200 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1874')
2026-04-30 10:31:33.410 | DEBUG    | __main__:load_electoral_data:53 - Processing election : ('legislative', '1876')
2026-04-30 10:31:33.638 | DEBUG    | __main__:load_electoral_dat

In [379]:
# Filter out unvalid codecommune — only France metropolitaine
election_datasets = election_datasets.filter(
    (
        pl.col("codecommune").str.slice(0, 2).is_between(pl.lit("01"), pl.lit("95"))
        | pl.col("codecommune").str.slice(0, 2).is_in(["2A", "2B"])
    )
)

In [380]:
election_datasets

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""AIN""","""01002""","""ABERGEMENT-DE-VAREY""",1848,"""legislative""","""leg1848""",133.0,133.0,138.0,40.599998,17.4,8.0,21.6,50.400002,62.0,76.0,58.0,72.0,null,null,false,false,138.0,1.0,0.0,0.29,0.13,0.06,0.16,0.37,0.45,0.55,0.42,0.52,null,null,0.0
"""01""","""AIN""","""01004""","""AMBERIEU-EN-BUGEY""",1848,"""legislative""","""leg1848""",807.0,807.0,840.0,245.7,105.3,55.0,130.2,303.79999,378.5,461.5,351.0,434.0,null,null,false,false,839.99999,1.0,0.0,0.29,0.13,0.07,0.16,0.36,0.45,0.55,0.42,0.52,null,null,0.0
"""01""","""AIN""","""01005""","""AMBERIEUX-EN-DOMBES""",1848,"""legislative""","""leg1848""",278.0,250.0,248.0,53.900002,23.1,33.0,41.400002,96.599998,93.5,154.5,77.0,138.0,null,null,false,false,248.000002,0.9,28.0,0.22,0.09,0.13,0.17,0.39,0.38,0.62,0.31,0.56,null,null,0.1
"""01""","""AIN""","""01006""","""AMBLEON""",1848,"""legislative""","""leg1848""",62.0,56.0,55.0,14.0,6.0,8.0,8.1000004,18.9,24.0,31.0,20.0,27.0,null,null,false,false,55.0,0.9,6.0,0.25,0.11,0.15,0.15,0.34,0.44,0.56,0.36,0.49,null,null,0.1
"""01""","""AIN""","""01007""","""AMBRONAY""",1848,"""legislative""","""leg1848""",413.0,413.0,430.0,124.6,53.400002,29.0,66.900002,156.10001,192.5,237.5,178.0,223.0,null,null,false,false,430.000014,1.0,0.0,0.29,0.12,0.07,0.16,0.36,0.45,0.55,0.41,0.52,null,null,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""95""","""VAL D'OISE""","""95676""","""Villers-en-Arthies""",2005,"""referendum""","""ref2005""",345.0,260.0,255.0,null,null,null,null,null,null,null,null,null,135.0,120.0,false,false,255.0,0.75,85.0,null,null,null,null,null,null,null,null,null,0.53,0.47,0.25
"""95""","""VAL D'OISE""","""95678""","""Villiers-Adam""",2005,"""referendum""","""ref2005""",572.0,461.0,449.0,null,null,null,null,null,null,null,null,null,201.0,248.0,false,false,449.0,0.81,111.0,null,null,null,null,null,null,null,null,null,0.45,0.55,0.19
"""95""","""VAL D'OISE""","""95680""","""Villiers-le-Bel""",2005,"""referendum""","""ref2005""",10857.0,6185.0,6061.0,null,null,null,null,null,null,null,null,null,2179.0,3882.0,false,false,6061.0,0.57,4672.0,null,null,null,null,null,null,null,null,null,0.36,0.64,0.43


In [381]:
# Remove election for which we have no information
cols_to_check = [
    "inscrits",
    "votants",
    "exprimes",
    "voteG",
    "voteCG",
    "voteC",
    "voteCD",
    "voteD",
    "voteTG",
    "voteTD",
    "voteGCG",
    "voteDCD",
    "voixOUI",
    "voixNON",
]

election_datasets = election_datasets.filter(
    ~pl.all_horizontal(pl.col(cols_to_check).is_null())
)
election_datasets.null_count()

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,24,174235,174235,174235,174235,174235,174235,174235,174235,174235,2314408,2314408,0,0,0,0,0,174235,174235,174235,174235,174235,174235,174235,174235,174235,2314408,2314408,0


In [382]:
cols_to_zero_inscrits = [
    "votants",
    "exprimes",
    "voteG",
    "voteCG",
    "voteC",
    "voteCD",
    "voteD",
    "voteTG",
    "voteTD",
    "voteGCG",
    "voteDCD",
    "voixOUI",
    "voixNON",
]

cols_to_zero_votants = [
    "exprimes",
    "voteG",
    "voteCG",
    "voteC",
    "voteCD",
    "voteD",
    "voteTG",
    "voteTD",
    "voteGCG",
    "voteDCD",
    "voixOUI",
    "voixNON",
]

election_datasets = election_datasets.with_columns(
    # Zero out cols when inscrits == 0
    [
        pl.when(pl.col("inscrits") == 0).then(0).otherwise(pl.col(c)).alias(c)
        for c in cols_to_zero_inscrits
    ]
    + [
        pl.when(pl.col("inscrits") == 0)
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("no_inscrits")
    ]
).with_columns(
    # Zero out cols when votants == 0
    [
        pl.when(pl.col("votants") == 0).then(0).otherwise(pl.col(c)).alias(c)
        for c in cols_to_zero_votants
    ]
    + [
        pl.when(pl.col("votants") == 0)
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("no_votants")
    ]
)
print(election_datasets.filter(pl.col("inscrits") < 0).height)
print(election_datasets.filter(pl.col("votants") < 0).height)

0
0


In [383]:
# Data quality checks on election data

# 1. No negative amount in vote columns
vote_cols = [
    "inscrits",
    "votants",
    "exprimes",
    "inscritsT2",
    "votantsT2",
    "exprimesT2",
    "voteG",
    "voteCG",
    "voteC",
    "voteCD",
    "voteD",
    "voteTG",
    "voteTD",
    "voteGCG",
    "voteDCD",
    "voixOUI",
    "voixNON",
]

df_negative = election_datasets.filter(
    pl.any_horizontal(
        *[pl.col(c) < 0 for c in vote_cols if c in election_datasets.columns]
    )
)
df_negative
# One outlier rows for leg1871juil, removing it

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""76""","""SEINE-MARITIME""","""76362""","""HEURTEAUVILLE""",1874,"""legislative""","""leg1871juil""",145.0,64.0,-7.0,-5.340741,0.0,0.0,-1.659259,0.0,-5.340741,-1.659259,-5.340741,-1.659259,null,null,false,false,-7.0,0.44,81.0,0.76,-0.0,-0.0,0.24,-0.0,0.76,0.24,0.76,0.24,null,null,0.56


In [385]:
election_datasets.filter(pl.col("inscrits") == 0)

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""AIN""","""01059""","""BRENAZ""",1848,"""legislative""","""leg1848""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""01""","""AIN""","""01091""","""CHATILLON-EN-MICHAILLE""",1848,"""legislative""","""leg1848""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""01""","""AIN""","""01097""","""CHAVORNAY""",1848,"""legislative""","""leg1848""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""01""","""AIN""","""01119""","""CORCELLES""",1848,"""legislative""","""leg1848""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""01""","""AIN""","""01122""","""CORMARANCHE-EN-BUGEY""",1848,"""legislative""","""leg1848""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""90""","""TERRITOIRE DE BELFORT""","""90040""","""ETUEFFONT-BAS""",1795,"""referendum""","""ref1795""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""90""","""TERRITOIRE DE BELFORT""","""90073""","""MOVAL""",1795,"""referendum""","""ref1795""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""90""","""TERRITOIRE DE BELFORT""","""90092""","""SALBERT""",1795,"""referendum""","""ref1795""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,true,true,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [386]:
election_datasets.filter(pl.col("inscrits") > 0).filter(pl.col("votants") == 0)

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""21""","""COTE-D'OR""","""21161""","""CHAUMONT-LE-BOIS""",1874,"""legislative""","""leg1871juil""",123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""21""","""COTE-D'OR""","""21198""","""CORROMBLES""",1874,"""legislative""","""leg1871juil""",167.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,167.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""21""","""COTE-D'OR""","""21272""","""FLEE""",1874,"""legislative""","""leg1871juil""",119.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,119.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""21""","""COTE-D'OR""","""21368""","""MAGNY-LES-VILLERS""",1874,"""legislative""","""leg1871juil""",80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""21""","""COTE-D'OR""","""21382""","""MARCILLY-OGNY""",1874,"""legislative""","""leg1871juil""",220.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,220.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""85""","""VENDEE""","""85197""","""SAINT-ANDRE-TREIZE-VOIES""",1795,"""referendum""","""ref1795""",209.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,209.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""85""","""VENDEE""","""85215""","""SAINT-FULGENT""",1795,"""referendum""","""ref1795""",152.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,152.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""85""","""VENDEE""","""85262""","""SAINT-PHILBERT-DE-BOUAINE""",1795,"""referendum""","""ref1795""",163.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,false,true,0.0,0.0,163.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [387]:
# Be careful :
# - Inscrits and votants are two columns that go in pairs and we have inscrits > votants
# - Exprimes and vote columns go together and their sum should be equal
# But we don't have a match between votants and exprimes !

In [388]:
# On outlier row
election_datasets.filter(pl.col("inscrits").round(0) < pl.col("votants").round(0))

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""60""","""Oise""","""60400""","""Le Mesnil-sur-Bulles""",2026,"""municipales""","""muni2026""",209.0,212.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,false,false,0.0,1.0,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0


In [389]:
# Check the coherence of the number of voix
# Ok very few desc only in 2022. We exclude municipales and referundum
election_datasets_check = (
    election_datasets.filter(~(pl.col("type").is_in(["municipales", "referendum"])))
    .with_columns(
        (
            pl.sum_horizontal(
                "voteG", "voteCG", "voteC", "voteCD", "voteD", "voixOUI", "voixNON"
            )
        ).alias("total_voix_5_blocs"),
        (
            pl.sum_horizontal(
                "voteG", "voteCG", "voteC", "voteCD", "voteD", "voixOUI", "voixNON"
            ).round(0)
            == pl.col("exprimes").round(0)
        ).alias("matching_voix_5_blocs"),
        (
            pl.sum_horizontal(
                "voteGCG", "voteC", "voteDCD", "voixOUI", "voixNON"
            ).round(0)
            == pl.col("exprimes").round(0)
        ).alias("matching_voix_3_blocs"),
        (
            pl.sum_horizontal("voteTG", "voteTD", "voixOUI", "voixNON").round(0)
            == pl.col("exprimes").round(0)
        ).alias("matching_voix_2_blocs"),
    )
    .with_columns(
        (pl.col("exprimes") - pl.col("total_voix_5_blocs")).alias("diff"),
        (pl.col("inscrits") - pl.col("total_voix_5_blocs")).alias("diff_inscrits"),
    )
)
print(election_datasets_check.height)
print(election_datasets_check.filter(pl.col("matching_voix_5_blocs") == False))
# Correction to have exprimes == sum_voix
election_datasets = election_datasets.with_columns(
    exprimes_=pl.sum_horizontal(
        "voteG", "voteCG", "voteC", "voteCD", "voteD", "voixOUI", "voixNON"
    )
)

2219210
shape: (4, 44)
┌─────┬──────────┬─────────────┬──────────────┬───┬─────────────┬─────────────┬──────┬─────────────┐
│ dep ┆ nomdep   ┆ codecommune ┆ nomcommune   ┆ … ┆ matching_vo ┆ matching_vo ┆ diff ┆ diff_inscri │
│ --- ┆ ---      ┆ ---         ┆ ---          ┆   ┆ ix_3_blocs  ┆ ix_2_blocs  ┆ ---  ┆ ts          │
│ str ┆ str      ┆ str         ┆ str          ┆   ┆ ---         ┆ ---         ┆ f64  ┆ ---         │
│     ┆          ┆             ┆              ┆   ┆ bool        ┆ bool        ┆      ┆ f64         │
╞═════╪══════════╪═════════════╪══════════════╪═══╪═════════════╪═════════════╪══════╪═════════════╡
│ 16  ┆ Charente ┆ 16029       ┆ Bardenac     ┆ … ┆ false       ┆ false       ┆ 2.0  ┆ 66.0        │
│ 27  ┆ Eure     ┆ 27322       ┆ La Haye-Malh ┆ … ┆ false       ┆ false       ┆ -1.0 ┆ 317.0       │
│     ┆          ┆             ┆ erbe         ┆   ┆             ┆             ┆      ┆             │
│ 34  ┆ Hérault  ┆ 34316       ┆ Usclas-du-Bo ┆ … ┆ false       ┆ fa

In [390]:
p_cols = [
    "ppar",
    "pvoteG",
    "pvoteCG",
    "pvoteC",
    "pvoteCD",
    "pvoteD",
    "pvoteTG",
    "pvoteTD",
    "pvoteGCG",
    "pvoteDCD",
    "pvoixOUI",
    "pvoixNON",
    "pabs",
]

election_datasets = (
    election_datasets.with_columns(
        ppar=pl.col("votants") / pl.col("inscrits"),
        abstentions=pl.col("inscrits") - pl.col("votants"),
        pvoteG=pl.col("voteG") / pl.col("exprimes_"),
        pvoteCG=pl.col("voteCG") / pl.col("exprimes_"),
        pvoteC=pl.col("voteC") / pl.col("exprimes_"),
        pvoteCD=pl.col("voteCD") / pl.col("exprimes_"),
        pvoteD=pl.col("voteD") / pl.col("exprimes_"),
        pvoteTG=pl.col("voteTG") / pl.col("exprimes_"),
        pvoteTD=pl.col("voteTD") / pl.col("exprimes_"),
        pvoteGCG=pl.col("voteGCG") / pl.col("exprimes_"),
        pvoteDCD=pl.col("voteDCD") / pl.col("exprimes_"),
        pvoixOUI=pl.col("voixOUI") / pl.col("exprimes_"),
        pvoixNON=pl.col("voixNON") / pl.col("exprimes_"),
    )
    .with_columns(
        pabs=pl.col("abstentions") / pl.col("inscrits"),
    )
    .with_columns(
        # Replace NaN (0/0) and inf (n/0) with 0
        pl.col(p_cols).fill_nan(0).clip(lower_bound=0, upper_bound=1.0)
    )
    .with_columns(pl.col(p_cols).round(2))
)

# Assert all p* columns are between 0 and 1
out_of_range = election_datasets.filter(
    pl.any_horizontal(*[(pl.col(c) < 0) | (pl.col(c) > 1) for c in p_cols])
)

out_of_range

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,voixOUI,voixNON,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoixOUI,pvoixNON,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [232]:
socio_economic_data = load_socio_economic_data()

2026-04-30 08:44:23.342 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : agesexcommunes
2026-04-30 08:44:45.064 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : menagescommunes
2026-04-30 08:44:45.237 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : crimesdelitscommunes
2026-04-30 08:44:48.802 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : cspcommunes
2026-04-30 08:45:21.421 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : empfoncommunes
2026-04-30 08:45:27.740 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : emploicommunes
2026-04-30 08:45:28.849 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : rsacommunes
2026-04-30 08:45:30.031 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : basesfiscalescommunes
2026-04-30 08:45:47.187 | DEBUG    | __main__:load_socio_economic_data:115 - Processing file : capitalimmobilier

In [235]:
# Check there is non inf or NaN : all replace by Null
nan_counts = socio_economic_data.drop("codecommune", "feature_name", "variable").select(
    pl.all().is_nan().sum()
)
print("NaN")
print(nan_counts)
print(nan_counts.sum_horizontal().item())

# Which columns have any inf?
inf_counts = socio_economic_data.drop("codecommune", "feature_name", "variable").select(
    pl.all().is_infinite().sum()
)
print("Inf")
print(inf_counts)
print(inf_counts.sum_horizontal().item())

NaN
shape: (1, 6)
┌─────┬──────┬─────┬──────┬───────┬────────────┐
│ raw ┆ year ┆ lag ┆ rank ┆ delta ┆ pct_change │
│ --- ┆ ---  ┆ --- ┆ ---  ┆ ---   ┆ ---        │
│ u32 ┆ u32  ┆ u32 ┆ u32  ┆ u32   ┆ u32        │
╞═════╪══════╪═════╪══════╪═══════╪════════════╡
│ 0   ┆ 0    ┆ 0   ┆ 0    ┆ 0     ┆ 0          │
└─────┴──────┴─────┴──────┴───────┴────────────┘
0
Inf
shape: (1, 6)
┌─────┬──────┬─────┬──────┬───────┬────────────┐
│ raw ┆ year ┆ lag ┆ rank ┆ delta ┆ pct_change │
│ --- ┆ ---  ┆ --- ┆ ---  ┆ ---   ┆ ---        │
│ u32 ┆ u32  ┆ u32 ┆ u32  ┆ u32   ┆ u32        │
╞═════╪══════╪═════╪══════╪═══════╪════════════╡
│ 0   ┆ 0    ┆ 0   ┆ 0    ┆ 0     ┆ 0          │
└─────┴──────┴─────┴──────┴───────┴────────────┘
0


In [236]:
result = socio_economic_data.group_by("feature_name").agg(
    pl.col("year").unique().sort()
)

In [237]:
result

feature_name,year
str,list[i64]
"""franatur""","[1962, 1975, 1990]"
"""nprim""",[2021]
"""paica""","[1960, 1961, … 2022]"
"""adm""","[1982, 1990, … 2016]"
"""pconjsign""","[1686, 1786, … 1905]"
…,…
"""peragri""","[1790, 1791, … 2022]"
"""recetteimpotslocauxratio""","[1881, 1911, … 2022]"
"""prixm2""","[2014, 2015, … 2022]"


In [238]:
from shapely.geometry import shape


def add_geographical_data():
    """Add geographical coordinates from GeoJSON file"""
    logger.info("Adding geographical data...")

    geo_data_path = data_path + "raw/geo_data/communes.geojson"

    fs = DataUtils._create_fs() if DataUtils._detect_s3(geo_data_path) else None
    if not DataUtils._exists(geo_data_path, fs):
        logger.warning(
            f"GeoJSON file not found at {geo_data_path}. Skipping geographical data."
        )
        return None

    # Load the GeoJSON map of French communes
    communes_geojson = DataLoader.load_geojson(geo_data_path)

    geo_data = []
    for feature in communes_geojson["features"]:
        commune_geom = shape(feature["geometry"])
        lat = float(commune_geom.centroid.x)
        long = float(commune_geom.centroid.y)
        codecommune = feature["properties"]["code"]
        geo_data.append({"codecommune": codecommune, "lat": lat, "long": long})

        # Handle PARIS / LYON / MARSEILLE
        if codecommune == "13055":
            for i in range(1, 16 + 1):
                if i < 10:
                    geo_data.append(
                        {"codecommune": f"1320{i}", "lat": lat, "long": long}
                    )
                else:
                    geo_data.append(
                        {"codecommune": f"132{i}", "lat": lat, "long": long}
                    )
        if codecommune == "69123":
            for i in range(1, 9 + 1):
                geo_data.append({"codecommune": f"6938{i}", "lat": lat, "long": long})
        if codecommune == "75056":
            for i in range(1, 20 + 1):
                if i < 10:
                    geo_data.append(
                        {"codecommune": f"7510{i}", "lat": lat, "long": long}
                    )
                else:
                    geo_data.append(
                        {"codecommune": f"751{i}", "lat": lat, "long": long}
                    )

    return pl.DataFrame(geo_data).sort("codecommune")

In [239]:
geo_data = add_geographical_data()

2026-04-30 08:48:23.970 | INFO     | __main__:add_geographical_data:5 - Adding geographical data...
2026-04-30 08:48:25.639 | DEBUG    | src.components.data_processing.data_loader:load_geojson:217 - Geojson loaded


In [240]:
geo_data = geo_data.with_columns(
    lat=pl.col("lat").round(4), long=pl.col("long").round(4)
)
geo_data

codecommune,lat,long
str,f64,f64
"""01001""",4.9258,46.1536
"""01002""",5.428,46.0096
"""01004""",5.3724,45.961
"""01005""",4.9119,45.9962
"""01006""",5.5946,45.7498
…,…,…
"""95676""",1.7306,49.0861
"""95678""",2.2389,49.0699
"""95680""",2.4039,49.0079


In [241]:
communes = (
    communes.join(geo_data, on="codecommune", how="left")
    # Step 1: fill from parent commune
    .join(
        geo_data.select(["codecommune", "lat", "long"]).rename(
            {"lat": "lat_parent", "long": "long_parent"}
        ),
        left_on="codecommune_parent",
        right_on="codecommune",
        how="left",
    )
    .with_columns(
        lat=pl.col("lat").fill_null(pl.col("lat_parent")),
        long=pl.col("long").fill_null(pl.col("long_parent")),
    )
    .drop(["lat_parent", "long_parent"])
    # Step 2: replace NaN with null so fill_null works on them too
    .with_columns(
        pl.col("lat").fill_nan(None),
        pl.col("long").fill_nan(None),
    )
    # Step 3: fill remaining nulls with department mean
    .with_columns(
        lat=pl.col("lat").fill_null(pl.col("lat").mean().over("DEP")),
        long=pl.col("long").fill_null(pl.col("long").mean().over("DEP")),
    )
)

In [242]:
# Communes d'outre mer
communes.filter(pl.col("lat").is_null()).get_column("DEP").value_counts()

DEP,count
str,u32
"""971""",32
"""976""",17
"""972""",34
"""974""",24
"""973""",22


In [243]:
def data_quality_checks(df):
    # Which columns have any NaN?
    nan_counts = df.select(pl.all().is_nan().sum())
    print("NaN")
    print(nan_counts)
    print(nan_counts.sum_horizontal().item())

    # Which columns have any Null?
    null_counts = df.select(pl.all().is_null().sum())
    print("Null")
    print(null_counts)
    print(null_counts.sum_horizontal().item())

    # Which columns have any inf?
    inf_counts = df.select(pl.all().is_infinite().sum())
    print("Inf")
    print(inf_counts)
    print(inf_counts.sum_horizontal().item())

In [306]:
def get_features_for_years(
    X,
    year=2022,
    feature_aug=["raw", "lag", "rank", "delta", "pct_change"],
    n_years_back=0,
):
    index_cols = ["codecommune"]

    # Base: pivot for the election year
    data = (
        X.filter(pl.col("year") == year)
        .pivot(
            on="feature_name",
            index=["year", "codecommune"],
            values=feature_aug,
            aggregate_function="first",
        )
        .sort("codecommune")
    )

    # Iteratively join each year back
    for i in range(1, n_years_back + 1):
        year_back = year - i
        df_back = (
            X.filter(pl.col("year") == year_back)
            .pivot(
                on="feature_name",
                index=["year", "codecommune"],
                values=feature_aug,
                aggregate_function="first",
            )
            .sort("codecommune")
        )

        new_cols = [
            c for c in df_back.columns if c not in data.columns and c not in index_cols
        ]

        data = data.join(
            df_back.select(index_cols + new_cols).rename(
                {c: f"{c}|minus_{i}" for c in new_cols}
            ),
            on=index_cols,
            how="left",
        )

    return data

In [315]:
X = get_features_for_years(
    socio_economic_data,
    year=2022,
    feature_aug=["raw", "lag", "rank", "delta", "pct_change"],
    n_years_back=0,
)

In [316]:
X

year,codecommune,raw_age,raw_agef,raw_ageh,raw_perage,raw_perprop014,raw_perprop60p,raw_perpropf,raw_pop,raw_popf,raw_popf014,raw_popf1539,raw_popf4059,raw_popf60p,raw_poph,raw_poph014,raw_poph1539,raw_poph4059,raw_poph60p,raw_prop014,raw_prop1539,raw_prop4059,raw_prop60p,raw_propf,raw_propf014,raw_propf1539,raw_propf4059,raw_propf60p,raw_agri,raw_aica,raw_aind,raw_cadr,raw_capi,raw_chom,raw_empl,raw_indp,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2022,""".""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2022,"""..""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2022,"""...""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2022,"""01001""",44.203548,43.147369,45.297142,0.740719,0.7814948,0.2082655,0.312094,747.0,380.0,118.0,35.0,142.0,85.0,367.0,42.0,86.0,175.0,64.0,0.2141901,0.161981,0.424364,0.199465,0.508701,0.7375,0.289256,0.44795,0.5704698,0.0,72.0,39.0,33.0,47.0,26.0,86.0,39.0,…,-0.416667,-0.421053,-0.233834,-0.642754,-0.44929,-0.415914,-0.420306,-0.427021,0.019512,0.02071,0.000515,0.001175,0.040843,0.016332,0.004792,-0.001773,0.0,0.008687,0.011398,0.009007,0.0,0.004016,0.001776,0.0,0.011484,0.008497,0.011481,0.0,0.0,0.0,-0.002321,-0.00156,0.002551,0.002467,0.0,0.0,0.0
2022,"""01002""",36.973824,27.685621,44.404388,0.159299,0.983913,0.333964,0.045994,288.0,128.0,58.0,8.0,61.0,1.0,160.0,28.0,63.0,4.0,65.0,0.2986111,0.246528,0.225694,0.229167,0.444444,0.674419,0.112676,0.938462,0.015152,0.0,48.0,39.0,9.0,9.0,9.0,66.0,39.0,…,-1.0,-1.0,0.046494,-0.69693,-0.67834,-1.0,-1.0,-1.0,0.016129,0.02,0.005309,0.00381,-0.005287,-0.001793,0.004792,0.005348,0.0,-0.033199,0.003869,-0.030328,0.0,-0.013889,-0.005319,0.0,-0.006554,0.001354,-0.006557,0.0,0.0,0.0,0.015146,0.019021,0.011407,-0.006909,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2022,"""95Z001""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,

Creation of a dataset for a given election : election_code

## Pipeline

In [543]:
election_code = "pres2017"

In [544]:
# Step 1: get all communes
communes

TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT,DEP_NUM,codecommune,codecommune_parent,lat,long,lat_right,long_right
str,str,i64,str,str,str,i64,str,str,str,str,i64,i64,str,str,f64,f64,f64,f64
"""COM""","""1001""",84,"""1""","""01D""","""12""",5,"""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""","""L'Abergement-Clémenciat""","""108""",null,1,"""01001""",null,4.925824,46.153595,4.9258,46.1536
"""COM""","""1002""",84,"""1""","""01D""","""11""",5,"""ABERGEMENT DE VAREY""","""Abergement-de-Varey""","""L'Abergement-de-Varey""","""101""",null,1,"""01002""",null,5.427993,46.009579,5.428,46.0096
"""COM""","""1004""",84,"""1""","""01D""","""11""",1,"""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""","""Ambérieu-en-Bugey""","""101""",null,1,"""01004""",null,5.372353,45.961045,5.3724,45.961
"""COM""","""1005""",84,"""1""","""01D""","""12""",1,"""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""","""Ambérieux-en-Dombes""","""122""",null,1,"""01005""",null,4.911941,45.996194,4.9119,45.9962
"""COM""","""1006""",84,"""1""","""01D""","""11""",1,"""AMBLEON""","""Ambléon""","""Ambléon""","""104""",null,1,"""01006""",null,5.594622,45.749777,5.5946,45.7498
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""COM""","""97613""",6,"""976""","""976R""",null,0,"""M TSANGAMOUJI""","""M'Tsangamouji""","""M'Tsangamouji""","""97613""",null,976,"""97613""",null,null,null,null,null
"""COM""","""97614""",6,"""976""","""976R""",null,1,"""OUANGANI""","""Ouangani""","""Ouangani""","""97610""",null,976,"""97614""",null,null,null,null,null
"""COM""","""97615""",6,"""976""","""976R""",null,0,"""PAMANDZI""","""Pamandzi""","""Pamandzi""","""97611""",null,976,"""97615""",null,null,null,null,null


In [577]:
# Step 2: get election results
election_results = election_datasets.filter(pl.col("election_code") == election_code)
election_results = election_results[
    [
        s.name
        for s in election_results
        if not (s.null_count() == election_results.height)
    ]
]
election_results

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""Ain""","""01001""","""L'ABERGEMENT-CLÃMENCIAT""",2017,"""presidentiel""","""pres2017""",598.0,506.0,495.0,67.800003,29.799999,119.8,110.8,166.8,157.5,337.5,97.600006,277.60001,false,false,495.000002,0.85,92.0,0.14,0.06,0.24,0.22,0.34,0.32,0.68,0.2,0.56,0.15
"""01""","""Ain""","""01002""","""L'ABERGEMENT-DE-VAREY""",2017,"""presidentiel""","""pres2017""",209.0,184.0,176.0,37.0,13.0,37.0,34.0,55.0,68.5,107.5,50.0,89.0,false,false,176.0,0.88,25.0,0.21,0.07,0.21,0.19,0.31,0.39,0.61,0.28,0.51,0.12
"""01""","""Ain""","""01004""","""AMBÃRIEU-EN-BUGEY""",2017,"""presidentiel""","""pres2017""",8586.0,6624.0,6452.0,1556.0,357.0,1345.0,1097.0,2097.0,2585.5,3866.5,1913.0,3194.0,false,false,6452.0,0.77,1962.0,0.24,0.06,0.21,0.17,0.33,0.4,0.6,0.3,0.5,0.23
"""01""","""Ain""","""01005""","""AMBÃRIEUX-EN-DOMBES""",2017,"""presidentiel""","""pres2017""",1172.0,957.0,933.0,142.2,38.200001,192.2,198.2,362.20001,276.5,656.5,180.39999,560.40002,false,false,933.000011,0.82,215.0,0.15,0.04,0.21,0.21,0.39,0.3,0.7,0.19,0.6,0.18
"""01""","""Ain""","""01006""","""AMBLÃON""",2017,"""presidentiel""","""pres2017""",99.0,79.0,77.0,22.200001,3.2,15.2,14.2,22.200001,33.0,44.0,25.400002,36.400002,false,false,77.000002,0.8,20.0,0.29,0.04,0.2,0.18,0.29,0.43,0.57,0.33,0.47,0.2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""95""","""Val-d'Oise""","""95676""","""VILLERS-EN-ARTHIES""",2017,"""presidentiel""","""pres2017""",382.0,339.0,334.0,69.400002,10.4,47.400002,110.4,96.400002,103.5,230.5,79.800003,206.8,false,false,334.000006,0.89,43.0,0.21,0.03,0.14,0.33,0.29,0.31,0.69,0.24,0.62,0.11
"""95""","""Val-d'Oise""","""95678""","""VILLIERS-ADAM""",2017,"""presidentiel""","""pres2017""",593.0,545.0,535.0,88.599998,28.6,125.6,140.60001,151.60001,180.0,355.0,117.2,292.20001,false,false,535.000018,0.92,48.0,0.17,0.05,0.23,0.26,0.28,0.34,0.66,0.22,0.55,0.08
"""95""","""Val-d'Oise""","""95680""","""VILLIERS-LE-BEL""",2017,"""presidentiel""","""pres2017""",13213.0,8821.0,8558.0,3084.6001,741.59998,1949.6,1122.6,1659.6,4801.0,3757.0,3826.2002,2782.2,false,false,8558.00008,0.67,4392.0,0.36,0.09,0.23,0.13,0.19,0.56,0.44,0.45,0.33,0.33


In [546]:
dataset = election_results.join(
    communes.select(
        "TYPECOM", "codecommune", "NCC", "NCCENR", "LIBELLE", "lat", "long"
    ),
    on="codecommune",
    how="left",
)
dataset = dataset.with_columns(
    lat=pl.col("lat").fill_null(pl.col("lat").mean().over("dep")),
    long=pl.col("long").fill_null(pl.col("long").mean().over("dep")),
)

In [547]:
# Step 3 : get relevant economic data
year = int(election_code[-4:])
X = get_features_for_years(
    socio_economic_data,
    year=year,
    feature_aug=["raw", "lag", "rank", "delta", "pct_change"],
    n_years_back=0,
)
X

year,codecommune,raw_age,raw_agef,raw_ageh,raw_perage,raw_perprop014,raw_perprop60p,raw_perpropf,raw_pop,raw_popf,raw_popf014,raw_popf1539,raw_popf4059,raw_popf60p,raw_poph,raw_poph014,raw_poph1539,raw_poph4059,raw_poph60p,raw_prop014,raw_prop1539,raw_prop4059,raw_prop60p,raw_propf,raw_propf014,raw_propf1539,raw_propf4059,raw_propf60p,raw_agri,raw_aica,raw_aind,raw_cadr,raw_capi,raw_chom,raw_empl,raw_indp,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2017,""".""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,"""..""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,"""...""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,"""01001""",41.607021,41.094738,42.116623,0.610999,0.788556,0.239692,0.171526,762.0,380.0,99.0,68.0,132.0,81.0,382.0,61.0,98.0,150.0,73.0,0.209974,0.217848,0.370079,0.2021,0.498688,0.61875,0.409639,0.468085,0.525974,3.0,73.0,26.0,47.0,91.0,26.0,85.0,23.0,…,0.333333,0.357143,-0.271209,-0.374402,-0.309456,0.335052,0.358892,0.337705,0.021448,0.019544,-0.004693,-0.001864,0.169245,0.037445,0.026189,-0.001754,0.026634,0.019446,0.029809,0.026915,-0.010583,0.03084,0.028602,0.000159,0.017498,0.016455,0.017372,-0.00863,0.006459,0.006459,0.007253,0.008832,0.011734,0.013037,0.0,0.0,0.0
2017,"""01002""",38.197773,34.541805,41.107628,0.252566,0.975373,0.504027,0.01776,264.0,117.0,39.0,26.0,35.0,17.0,147.0,30.0,56.0,13.0,48.0,0.261364,0.310606,0.181818,0.246212,0.443182,0.565217,0.317073,0.729167,0.261538,0.0,37.0,23.0,14.0,28.0,6.0,36.0,23.0,…,-0.5,-0.333333,-0.013676,-0.717598,-0.431452,-0.52621,-0.36828,-0.371274,0.018018,0.022472,0.005421,0.004375,0.295758,0.028548,0.037108,0.005618,0.03937,0.020577,0.020107,0.022118,-0.0182,0.022,0.031925,-0.001585,0.008772,0.019739,0.008647,-0.010358,0.017167,0.017167,0.039427,0.041133,0.020576,0.000794,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2017,"""95Z001""",null,null,null,null,null,null,null,null,null,null,null,null,null,

In [548]:
dataset = dataset.join(X.drop("year"), on="codecommune", how="left")

In [549]:
dataset.sort("codecommune")

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""Ain""","""01001""","""L'ABERGEMENT-CLÃMENCIAT""",2017,"""presidentiel""","""pres2017""",598.0,506.0,495.0,67.800003,29.799999,119.8,110.8,166.8,157.5,337.5,97.600006,277.60001,false,false,495.000002,0.85,92.0,0.14,0.06,0.24,0.22,0.34,0.32,0.68,0.2,0.56,0.15,"""COM""","""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""",…,0.333333,0.357143,-0.271209,-0.374402,-0.309456,0.335052,0.358892,0.337705,0.021448,0.019544,-0.004693,-0.001864,0.169245,0.037445,0.026189,-0.001754,0.026634,0.019446,0.029809,0.026915,-0.010583,0.03084,0.028602,0.000159,0.017498,0.016455,0.017372,-0.00863,0.006459,0.006459,0.007253,0.008832,0.011734,0.013037,0.0,0.0,0.0
"""01""","""Ain""","""01002""","""L'ABERGEMENT-DE-VAREY""",2017,"""presidentiel""","""pres2017""",209.0,184.0,176.0,37.0,13.0,37.0,34.0,55.0,68.5,107.5,50.0,89.0,false,false,176.0,0.88,25.0,0.21,0.07,0.21,0.19,0.31,0.39,0.61,0.28,0.51,0.12,"""COM""","""ABERGEMENT DE VAREY""","""Abergement-de-Varey""",…,-0.5,-0.333333,-0.013676,-0.717598,-0.431452,-0.52621,-0.36828,-0.371274,0.018018,0.022472,0.005421,0.004375,0.295758,0.028548,0.037108,0.005618,0.03937,0.020577,0.020107,0.022118,-0.0182,0.022,0.031925,-0.001585,0.008772,0.019739,0.008647,-0.010358,0.017167,0.017167,0.039427,0.041133,0.020576,0.000794,0.0,0.0,0.0
"""01""","""Ain""","""01004""","""AMBÃRIEU-EN-BUGEY""",2017,"""presidentiel""","""pres2017""",8586.0,6624.0,6452.0,1556.0,357.0,1345.0,1097.0,2097.0,2585.5,3866.5,1913.0,3194.0,false,false,6452.0,0.77,1962.0,0.24,0.06,0.21,0.17,0.33,0.4,0.6,0.3,0.5,0.23,"""COM""","""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""",…,-0.156156,-0.149813,-0.008131,-0.059251,0.028019,-0.144252,-0.137819,-0.131932,0.017128,0.017712,0.000121,0.000574,0.277659,0.024476,0.020818,0.004553,0.019423,0.029686,0.023589,-0.003754,-0.026963,0.017954,0.016793,0.001962,0.004779,0.004786,0.001252,-0.006843,0.001191,-0.000714,-0.001907,-0.001157,-0.001635,-0.003516,0.0,0.0,0.0
"""01""","""Ain""","""01005""","""AMBÃRIEUX-EN-DOMBES""",2017,"""presidentiel""","""pres2017""",1172.0,957.0,933.0,142.2,38.200001,192.2,198.2,362.20001,276.5,656.5,180.39999,560.40002,false,false,933.000011,0.82,215.0,0.15,0.04,0.21,0.21,0.39,0.3,0.7,0.19,0.6,0.18,"""COM""","""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""",…,0.162162,0.169492,-0.0081,-0.020578,0.06939,0.129823,0.136948,0.136499,0.010526,0.011468,-0.000191,0.000932,0.24467,0.039823,0.039492,0.017174,0.030162,0.034834,0.021894,0.036468,0.001876,0.033203,0.022547,0.009654,0.019831,0.010471,0.019704,0.000782,0.019505,0.019505,0.002543,0.00272,0.010772,0.020764,0.0,0.0,0.0
"""01""","""Ain""","""01006""","""AMBLÃON""",2017,"""presiden

In [550]:
dataset.null_count()

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,114,114,114,…,6586,6586,452,452,2226,6914,6914,6914,59,66,26,67,26,26,245,180,147,0,252,479,219,244,252,219,0,252,638,219,219,628,479,479,479,534,480,480,480


In [551]:
threshold = 0.4

cols_to_keep = [s.name for s in dataset if s.null_count() / dataset.height < threshold]
cols_dropped = [s.name for s in dataset if s.null_count() / dataset.height >= threshold]

print(f"✅ Kept {len(cols_to_keep)} columns.")
print(f"❌ Dropped {len(cols_dropped)} columns (>= {int(threshold * 100)}% nulls):")
for col in cols_dropped:
    null_ratio = (
        dataset.null_count()[col][0] / dataset.height
    )  # note: already dropped, use original if needed
print(cols_dropped)
dataset = dataset[cols_to_keep]

✅ Kept 825 columns.
❌ Dropped 45 columns (>= 40% nulls):
['raw_misf', 'raw_mmoyfortune', 'raw_nfoyisf', 'raw_permisf', 'raw_perpisf', 'raw_pisf', 'lag_nfoyerrsa', 'lag_nrsa', 'lag_perrsa', 'lag_prsa', 'lag_misf', 'lag_mmoyfortune', 'lag_nfoyisf', 'lag_permisf', 'lag_perpisf', 'lag_pisf', 'rank_misf', 'rank_mmoyfortune', 'rank_nfoyisf', 'rank_permisf', 'rank_perpisf', 'rank_pisf', 'delta_nfoyerrsa', 'delta_nrsa', 'delta_perrsa', 'delta_prsa', 'delta_misf', 'delta_mmoyfortune', 'delta_nfoyisf', 'delta_permisf', 'delta_perpisf', 'delta_pisf', 'pct_change_agri', 'pct_change_pagri', 'pct_change_nfoyerrsa', 'pct_change_nrsa', 'pct_change_perrsa', 'pct_change_prsa', 'pct_change_propappartement', 'pct_change_misf', 'pct_change_mmoyfortune', 'pct_change_nfoyisf', 'pct_change_permisf', 'pct_change_perpisf', 'pct_change_pisf']


In [552]:
dataset.null_count()

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,114,114,114,…,6586,6586,452,452,2226,6914,6914,6914,59,66,26,67,26,26,245,180,147,0,252,479,219,244,252,219,0,252,638,219,219,628,479,479,479,534,480,480,480


In [553]:
dataset.null_count().max_horizontal().item()

9987

In [554]:
null_counts = dataset.null_count()
max_val = null_counts.max_horizontal().item()

cols_with_max_nulls = [
    col
    for col, count in zip(null_counts.columns, null_counts.row(0))
    if count == max_val
]
print(cols_with_max_nulls)

['pct_change_pchom']


In [555]:
dataset.filter(pl.col("raw_age").is_null()).select(
    "pvoteG", "nomcommune", "codecommune", "raw_age"
).to_pandas().tail(100)

,pvoteG,nomcommune,codecommune,raw_age
0,0.15,DOMMARTIN,01144,NaN
1,0.16,LES COSTES,05043,NaN
2,0.20,MONTMORIN,05088,NaN
3,0.17,SAINT-EUSÃBE-EN-CHAMPSAUR,05141,NaN
4,0.08,SAINTE-MARIE,05150,NaN
...,...,...,...,...
91,0.19,SAINT-LUCIEN,76601,NaN
92,0.30,BELLEVILLE,79033,NaN
93,0.24,BOISSEROLLES,79039,NaN
94,0.34,SAINT-ETIENNE-LA-CIGOGNE,79247,NaN


In [556]:
meta_cols = [
    "dep",
    "nomdep",
    "codecommune",
    "nomcommune",
    "year",
    "type",
    "election_code",
    "inscrits",
    "votants",
    "exprimes",
    "voteG",
    "voteCG",
    "voteC",
    "voteCD",
    "voteD",
    "voteTG",
    "voteTD",
    "voteGCG",
    "voteDCD",
    "no_inscrits",
    "no_votants",
    "exprimes_",
    "ppar",
    "abstentions",
    "pvoteG",
    "pvoteCG",
    "pvoteC",
    "pvoteCD",
    "pvoteD",
    "pvoteTG",
    "pvoteTD",
    "pvoteGCG",
    "pvoteDCD",
    "pabs",
    "TYPECOM",
    "NCC",
    "NCCENR",
    "LIBELLE",
    "lat",
    "long",
]
features = [col for col in dataset.columns if col not in meta_cols]
print(features)

['raw_age', 'raw_agef', 'raw_ageh', 'raw_perage', 'raw_perprop014', 'raw_perprop60p', 'raw_perpropf', 'raw_pop', 'raw_popf', 'raw_popf014', 'raw_popf1539', 'raw_popf4059', 'raw_popf60p', 'raw_poph', 'raw_poph014', 'raw_poph1539', 'raw_poph4059', 'raw_poph60p', 'raw_prop014', 'raw_prop1539', 'raw_prop4059', 'raw_prop60p', 'raw_propf', 'raw_propf014', 'raw_propf1539', 'raw_propf4059', 'raw_propf60p', 'raw_agri', 'raw_aica', 'raw_aind', 'raw_cadr', 'raw_capi', 'raw_chom', 'raw_empl', 'raw_indp', 'raw_ouem', 'raw_ouvr', 'raw_pact', 'raw_pagri', 'raw_paica', 'raw_paind', 'raw_pcadr', 'raw_pcapi', 'raw_pchom', 'raw_pempl', 'raw_peragri', 'raw_peraica', 'raw_peraind', 'raw_percadr', 'raw_percapi', 'raw_perchom', 'raw_perempl', 'raw_perindp', 'raw_perouem', 'raw_perouvr', 'raw_perpint', 'raw_pindp', 'raw_pint', 'raw_pouem', 'raw_pouvr', 'raw_ppint', 'raw_nfoyerrsa', 'raw_nrsa', 'raw_perrsa', 'raw_prsa', 'raw_basefonciere', 'raw_basefonciereratio', 'raw_basefoncieretot', 'raw_basehabitation', '

In [557]:
dataset

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""Ain""","""01001""","""L'ABERGEMENT-CLÃMENCIAT""",2017,"""presidentiel""","""pres2017""",598.0,506.0,495.0,67.800003,29.799999,119.8,110.8,166.8,157.5,337.5,97.600006,277.60001,false,false,495.000002,0.85,92.0,0.14,0.06,0.24,0.22,0.34,0.32,0.68,0.2,0.56,0.15,"""COM""","""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""",…,0.333333,0.357143,-0.271209,-0.374402,-0.309456,0.335052,0.358892,0.337705,0.021448,0.019544,-0.004693,-0.001864,0.169245,0.037445,0.026189,-0.001754,0.026634,0.019446,0.029809,0.026915,-0.010583,0.03084,0.028602,0.000159,0.017498,0.016455,0.017372,-0.00863,0.006459,0.006459,0.007253,0.008832,0.011734,0.013037,0.0,0.0,0.0
"""01""","""Ain""","""01002""","""L'ABERGEMENT-DE-VAREY""",2017,"""presidentiel""","""pres2017""",209.0,184.0,176.0,37.0,13.0,37.0,34.0,55.0,68.5,107.5,50.0,89.0,false,false,176.0,0.88,25.0,0.21,0.07,0.21,0.19,0.31,0.39,0.61,0.28,0.51,0.12,"""COM""","""ABERGEMENT DE VAREY""","""Abergement-de-Varey""",…,-0.5,-0.333333,-0.013676,-0.717598,-0.431452,-0.52621,-0.36828,-0.371274,0.018018,0.022472,0.005421,0.004375,0.295758,0.028548,0.037108,0.005618,0.03937,0.020577,0.020107,0.022118,-0.0182,0.022,0.031925,-0.001585,0.008772,0.019739,0.008647,-0.010358,0.017167,0.017167,0.039427,0.041133,0.020576,0.000794,0.0,0.0,0.0
"""01""","""Ain""","""01004""","""AMBÃRIEU-EN-BUGEY""",2017,"""presidentiel""","""pres2017""",8586.0,6624.0,6452.0,1556.0,357.0,1345.0,1097.0,2097.0,2585.5,3866.5,1913.0,3194.0,false,false,6452.0,0.77,1962.0,0.24,0.06,0.21,0.17,0.33,0.4,0.6,0.3,0.5,0.23,"""COM""","""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""",…,-0.156156,-0.149813,-0.008131,-0.059251,0.028019,-0.144252,-0.137819,-0.131932,0.017128,0.017712,0.000121,0.000574,0.277659,0.024476,0.020818,0.004553,0.019423,0.029686,0.023589,-0.003754,-0.026963,0.017954,0.016793,0.001962,0.004779,0.004786,0.001252,-0.006843,0.001191,-0.000714,-0.001907,-0.001157,-0.001635,-0.003516,0.0,0.0,0.0
"""01""","""Ain""","""01005""","""AMBÃRIEUX-EN-DOMBES""",2017,"""presidentiel""","""pres2017""",1172.0,957.0,933.0,142.2,38.200001,192.2,198.2,362.20001,276.5,656.5,180.39999,560.40002,false,false,933.000011,0.82,215.0,0.15,0.04,0.21,0.21,0.39,0.3,0.7,0.19,0.6,0.18,"""COM""","""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""",…,0.162162,0.169492,-0.0081,-0.020578,0.06939,0.129823,0.136948,0.136499,0.010526,0.011468,-0.000191,0.000932,0.24467,0.039823,0.039492,0.017174,0.030162,0.034834,0.021894,0.036468,0.001876,0.033203,0.022547,0.009654,0.019831,0.010471,0.019704,0.000782,0.019505,0.019505,0.002543,0.00272,0.010772,0.020764,0.0,0.0,0.0
"""01""","""Ain""","""01006""","""AMBLÃON""",2017,"""presiden

In [558]:
dataset = dataset.with_columns(
    [
        pl.col(c)
        .fill_null(pl.col(c).mean().over("dep"))
        .fill_null(pl.col(c).mean())  # Fallback to all
        .alias(c)
        for c in features
    ]
)

In [559]:
# We need to remove col with no variation

# Compute std for all feature columns
std_df = dataset.select(features).std()

# Find columns with variation (std > 0 and not null)
cols_with_variation = [
    col
    for col, std_val in zip(std_df.columns, std_df.row(0))
    if std_val is not None and std_val > 0
]

# Report dropped columns
cols_dropped = [
    col
    for col, std_val in zip(std_df.columns, std_df.row(0))
    if std_val is None or std_val == 0
]

print(f"✅ Kept {len(cols_with_variation)} columns with variation.")
print(f"❌ Dropped {len(cols_dropped)} columns with no variation (std == 0 or null):")
for col in cols_dropped:
    print(f"   - {col}")

# Apply the filter
dataset = dataset.select(meta_cols + cols_with_variation)

✅ Kept 785 columns with variation.
❌ Dropped 0 columns with no variation (std == 0 or null):


In [560]:
data_quality_checks(dataset.drop(cs.string() | cs.boolean()))

NaN
shape: (1, 813)
┌──────┬──────────┬─────────┬──────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ year ┆ inscrits ┆ votants ┆ exprimes ┆ … ┆ pct_change_ ┆ pct_change_ ┆ pct_change_ ┆ pct_change_ │
│ ---  ┆ ---      ┆ ---     ┆ ---      ┆   ┆ electeurs   ┆ vbbm        ┆ vbbmpauvres ┆ vbbmpauvres │
│ u32  ┆ u32      ┆ u32     ┆ u32      ┆   ┆ ---         ┆ ---         ┆ riches      ┆ richesca…   │
│      ┆          ┆         ┆          ┆   ┆ u32         ┆ u32         ┆ ---         ┆ ---         │
│      ┆          ┆         ┆          ┆   ┆             ┆             ┆ u32         ┆ u32         │
╞══════╪══════════╪═════════╪══════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 0    ┆ 0        ┆ 0       ┆ 0        ┆ … ┆ 0           ┆ 0           ┆ 0           ┆ 0           │
└──────┴──────────┴─────────┴──────────┴───┴─────────────┴─────────────┴─────────────┴─────────────┘
0
Null
shape: (1, 813)
┌──────┬──────────┬─────────┬──────────┬───┬────

In [561]:
dataset.null_count().max_horizontal()

max
u32
114


In [562]:
null_counts = dataset.null_count()
max_val = null_counts.max_horizontal().item()

cols_with_max_nulls = [
    col
    for col, count in zip(null_counts.columns, null_counts.row(0))
    if count == max_val
]
print(cols_with_max_nulls)

['TYPECOM', 'NCC', 'NCCENR', 'LIBELLE']


In [ ]:
# dataset.write_csv('Y.csv')

In [567]:
dataset

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,pct_change_immnatur,pct_change_natur,pct_change_peretr,pct_change_perimmigre,pct_change_pimmigre,pct_change_pimmnatur,pct_change_pnatur,pct_change_pnaturfra,pct_change_nlogement,pct_change_npropri,pct_change_perpropri,pct_change_ppropri,pct_change_perpibratio,pct_change_pibratio,pct_change_pibtot,pct_change_nadult,pct_change_nfoyer,pct_change_perrev,pct_change_perrevadu,pct_change_perrevagglo,pct_change_perrevfoy,pct_change_revmoy,pct_change_revmoyadu,pct_change_revmoyfoy,pct_change_revratio,pct_change_revratioadu,pct_change_revratioagglo,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo,pct_change_peragglo,pct_change_percommu,pct_change_popagglo,pct_change_electeurs,pct_change_vbbm,pct_change_vbbmpauvresriches,pct_change_vbbmpauvresrichescap
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""Ain""","""01001""","""L'ABERGEMENT-CLÃMENCIAT""",2017,"""presidentiel""","""pres2017""",598.0,506.0,495.0,67.800003,29.799999,119.8,110.8,166.8,157.5,337.5,97.600006,277.60001,false,false,495.000002,0.85,92.0,0.14,0.06,0.24,0.22,0.34,0.32,0.68,0.2,0.56,0.15,"""COM""","""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""",…,0.333333,0.357143,-0.271209,-0.374402,-0.309456,0.335052,0.358892,0.337705,0.021448,0.019544,-0.004693,-0.001864,0.169245,0.037445,0.026189,-0.001754,0.026634,0.019446,0.029809,0.026915,-0.010583,0.03084,0.028602,0.000159,0.017498,0.016455,0.017372,-0.00863,0.006459,0.006459,0.007253,0.008832,0.011734,0.013037,0.0,0.0,0.0
"""01""","""Ain""","""01002""","""L'ABERGEMENT-DE-VAREY""",2017,"""presidentiel""","""pres2017""",209.0,184.0,176.0,37.0,13.0,37.0,34.0,55.0,68.5,107.5,50.0,89.0,false,false,176.0,0.88,25.0,0.21,0.07,0.21,0.19,0.31,0.39,0.61,0.28,0.51,0.12,"""COM""","""ABERGEMENT DE VAREY""","""Abergement-de-Varey""",…,-0.5,-0.333333,-0.013676,-0.717598,-0.431452,-0.52621,-0.36828,-0.371274,0.018018,0.022472,0.005421,0.004375,0.295758,0.028548,0.037108,0.005618,0.03937,0.020577,0.020107,0.022118,-0.0182,0.022,0.031925,-0.001585,0.008772,0.019739,0.008647,-0.010358,0.017167,0.017167,0.039427,0.041133,0.020576,0.000794,0.0,0.0,0.0
"""01""","""Ain""","""01004""","""AMBÃRIEU-EN-BUGEY""",2017,"""presidentiel""","""pres2017""",8586.0,6624.0,6452.0,1556.0,357.0,1345.0,1097.0,2097.0,2585.5,3866.5,1913.0,3194.0,false,false,6452.0,0.77,1962.0,0.24,0.06,0.21,0.17,0.33,0.4,0.6,0.3,0.5,0.23,"""COM""","""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""",…,-0.156156,-0.149813,-0.008131,-0.059251,0.028019,-0.144252,-0.137819,-0.131932,0.017128,0.017712,0.000121,0.000574,0.277659,0.024476,0.020818,0.004553,0.019423,0.029686,0.023589,-0.003754,-0.026963,0.017954,0.016793,0.001962,0.004779,0.004786,0.001252,-0.006843,0.001191,-0.000714,-0.001907,-0.001157,-0.001635,-0.003516,0.0,0.0,0.0
"""01""","""Ain""","""01005""","""AMBÃRIEUX-EN-DOMBES""",2017,"""presidentiel""","""pres2017""",1172.0,957.0,933.0,142.2,38.200001,192.2,198.2,362.20001,276.5,656.5,180.39999,560.40002,false,false,933.000011,0.82,215.0,0.15,0.04,0.21,0.21,0.39,0.3,0.7,0.19,0.6,0.18,"""COM""","""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""",…,0.162162,0.169492,-0.0081,-0.020578,0.06939,0.129823,0.136948,0.136499,0.010526,0.011468,-0.000191,0.000932,0.24467,0.039823,0.039492,0.017174,0.030162,0.034834,0.021894,0.036468,0.001876,0.033203,0.022547,0.009654,0.019831,0.010471,0.019704,0.000782,0.019505,0.019505,0.002543,0.00272,0.010772,0.020764,0.0,0.0,0.0
"""01""","""Ain""","""01006""","""AMBLÃON""",2017,"""presiden

1. Handle PLM — responsible for a lot of nulls
2. (Drop columns based on a threshold)
3. Input with the departemental mean (local mean)

Handle features not available for all years (merge with pivot of previous years?)

## Stacking

In [566]:
X = pl.scan_csv("X.csv", infer_schema_length=40000).collect()
Y = pl.scan_csv("Y.csv", infer_schema_length=40000).collect()

In [573]:
set(X.columns) - set(Y.columns)

set()

In [574]:
set(Y.columns) - set(X.columns)

{'delta_basefonciere',
 'delta_basefonciereratio',
 'delta_basefoncieretot',
 'delta_basehabitation',
 'delta_basehabitationratio',
 'delta_basehabitationtot',
 'delta_baseimpotslocaux',
 'delta_baseimpotslocauxratio',
 'delta_baseimpotslocauxtot',
 'delta_nfoyer',
 'delta_perrevfoy',
 'delta_recette',
 'delta_recettefonciere',
 'delta_recettefonciereratio',
 'delta_recettefoncieretot',
 'delta_recettehabitation',
 'delta_recettehabitationratio',
 'delta_recettehabitationtot',
 'delta_recetteimpotslocaux',
 'delta_recetteimpotslocauxratio',
 'delta_recetteimpotslocauxtot',
 'delta_recetteratio',
 'delta_recettetot',
 'delta_revmoyfoy',
 'delta_revratiofoy',
 'delta_revtot',
 'delta_revtotagglo',
 'delta_tauxfoncier',
 'delta_tauxfoncierratio',
 'delta_tauxhabitation',
 'delta_tauxhabitationratio',
 'delta_tauximpotslocaux',
 'delta_tauximpotslocauxratio',
 'lag_basefonciere',
 'lag_basefonciereratio',
 'lag_basefoncieretot',
 'lag_basehabitation',
 'lag_basehabitationratio',
 'lag_base

In [572]:
len(set(X.columns).intersection(set(Y.columns)))

676

In [575]:
result = pl.concat([X, Y], how="diagonal")

In [576]:
result

dep,nomdep,codecommune,nomcommune,year,type,election_code,inscrits,votants,exprimes,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,no_inscrits,no_votants,exprimes_,ppar,abstentions,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pabs,TYPECOM,NCC,NCCENR,…,delta_revmoyfoy,delta_revratiofoy,delta_revtot,delta_revtotagglo,pct_change_basefonciere,pct_change_basefonciereratio,pct_change_basefoncieretot,pct_change_basehabitation,pct_change_basehabitationratio,pct_change_basehabitationtot,pct_change_baseimpotslocaux,pct_change_baseimpotslocauxratio,pct_change_baseimpotslocauxtot,pct_change_recette,pct_change_recettefonciere,pct_change_recettefonciereratio,pct_change_recettefoncieretot,pct_change_recettehabitation,pct_change_recettehabitationratio,pct_change_recettehabitationtot,pct_change_recetteimpotslocaux,pct_change_recetteimpotslocauxratio,pct_change_recetteimpotslocauxtot,pct_change_recetteratio,pct_change_recettetot,pct_change_tauxfoncier,pct_change_tauxfoncierratio,pct_change_tauxhabitation,pct_change_tauxhabitationratio,pct_change_tauximpotslocaux,pct_change_tauximpotslocauxratio,pct_change_nfoyer,pct_change_perrevfoy,pct_change_revmoyfoy,pct_change_revratiofoy,pct_change_revtot,pct_change_revtotagglo
str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""01""","""Ain""","""01001""","""L'ABERGEMENT-CLÃMENCIAT""",2022,"""presidentiel""","""pres2022""",645.0,537.0,520.0,81.599998,38.599998,153.60001,29.6,216.60001,197.0,323.0,120.2,246.20001,false,false,520.000016,0.83,108.0,0.16,0.07,0.3,0.06,0.42,0.38,0.62,0.23,0.47,0.17,"""COM""","""ABERGEMENT CLEMENCIAT""","""Abergement-Clémenciat""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""01""","""Ain""","""01002""","""L'ABERGEMENT-DE-VAREY""",2022,"""presidentiel""","""pres2022""",213.0,175.0,171.0,55.0,15.0,52.0,10.0,39.0,96.0,75.0,70.0,49.0,false,false,171.0,0.82,38.0,0.32,0.09,0.3,0.06,0.23,0.56,0.44,0.41,0.29,0.18,"""COM""","""ABERGEMENT DE VAREY""","""Abergement-de-Varey""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""01""","""Ain""","""01004""","""AMBÃRIEU-EN-BUGEY""",2022,"""presidentiel""","""pres2022""",8765.0,6687.0,6553.0,1893.4,434.39999,1488.4,344.39999,2392.3999,3072.0,3481.0,2327.8,2736.7998,false,false,6552.99988,0.76,2078.0,0.29,0.07,0.23,0.05,0.37,0.47,0.53,0.36,0.42,0.24,"""COM""","""AMBERIEU EN BUGEY""","""Ambérieu-en-Bugey""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""01""","""Ain""","""01005""","""AMBÃRIEUX-EN-DOMBES""",2022,"""presidentiel""","""pres2022""",1282.0,1048.0,1028.0,170.8,59.799999,283.79999,52.799999,460.79999,372.5,655.5,230.60001,513.59998,false,false,1027.999978,0.82,234.0,0.17,0.06,0.28,0.05,0.45,0.36,0.64,0.22,0.5,0.18,"""COM""","""AMBERIEUX EN DOMBES""","""Ambérieux-en-Dombes""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""01""","""Ain""","""01006""","""AMBLÃON""",2022,"""presidentiel""","""pres2022""",103.0,80.0,77.0,21.4,4.4000001,21.4,9.3999996,20.4,36.5,40.5,25.799999,29.799999,false,false,77.0,0.78,23.0,0.28,0.06,0.28,0.12,0.26,0.47,0.53,0.34,0.39,0.22,"""COM""","""AMBLEON""","""Ambléon""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,nul